# Monthly Hazelnut Price — Model Findings

Three approaches tested, in order of complexity. Key question: what explains monthly hazelnut USD price returns without using production data?

In [ ]:
import sys, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV

from monthly_regression import build_monthly_dataset
from tobb_price_regression import _ols

plt.rcParams.update({'figure.dpi': 130, 'font.size': 9})

# Base dataset: monthly TOBB VWAP returns + 16 financial features
tobb_m  = build_monthly_dataset()
exclude = {'ret_vwap_try','ret_tryusd','ret_vwap_usd','ret_tur'}
feat_cols = [c for c in tobb_m.columns
             if c.startswith('ret_') and c not in exclude
             and tobb_m[c].notna().mean() >= 0.7]

base = tobb_m[['month','ret_vwap_usd'] + feat_cols].dropna()
y      = base['ret_vwap_usd'].values
X_raw  = base[feat_cols].values
ones   = np.ones(len(y))

print(f'n={len(base)} months  ({base["month"].iloc[0]} – {base["month"].iloc[-1]})')
print(f'16 features: {feat_cols}')

## 1 · Why Lasso→Ridge fails

All 16 features are monthly log returns of equities and commodity futures. The problem is **multicollinearity** — they all move together (driven by the same underlying commodity/risk cycle). Lasso sees 16 features that are all moderately correlated with `y`, but none is clearly dominant over the others after accounting for the rest. So it zeros *all* of them.

The correlation heatmap below shows why: most features are 0.3–0.6 correlated with each other.

In [ ]:
# Show feature-feature correlation heatmap
feat_df = base[feat_cols]
corr    = feat_df.corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Heatmap
ax = axes[0]
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(feat_cols)))
ax.set_yticks(range(len(feat_cols)))
labels = [c.replace('ret_','') for c in feat_cols]
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(labels, fontsize=7)
ax.set_title('Feature-feature correlations (multicollinearity)')
plt.colorbar(im, ax=ax)

# Bivariate R² with y
ax = axes[1]
bivars = {}
for i, f in enumerate(feat_cols):
    m = _ols(y, np.c_[ones, X_raw[:,i]], ['int', f], f)
    bivars[f.replace('ret_','')] = m.rsquared
bdf = pd.Series(bivars).sort_values(ascending=True)
colors = ['steelblue' if v > 0.05 else 'lightgray' for v in bdf.values]
ax.barh(bdf.index, bdf.values, color=colors)
ax.axvline(0, color='black', lw=0.5)
ax.set_xlabel('Bivariate R² with hazelnut USD return')
ax.set_title('Each feature alone (blue = R²>0.05)')

plt.tight_layout()
plt.show()

# Confirm Lasso zeros everything
scaler = StandardScaler()
X_s = scaler.fit_transform(X_raw)
lasso = LassoCV(cv=5, max_iter=20000, random_state=42)
lasso.fit(X_s, y)
survivors = [feat_cols[i] for i,c in enumerate(lasso.coef_) if abs(c) > 1e-8]
print(f'Lasso alpha={lasso.alpha_:.4f}')
print(f'Survivors: {survivors if survivors else "NONE — all features zeroed"}')
print(f'\nExplanation: features have high bivariate signal but zero each other out')
print(f'because they are all proxies for the same underlying factor.')

## 2 · PCA: extract the underlying factors first

Instead of regressing on the 16 correlated features directly, PCA rotates them into **orthogonal** (uncorrelated) components. Each component is a blend of the original features. Then we regress hazelnut returns on those components — no multicollinearity by construction.

- **PC1** (32% of feature variance): commodity / risk-on cycle. Bunge, SPY, soy, corn, MDLZ all load positively; VIX loads negatively. When global risk appetite is high → commodity prices rise → hazelnut USD price rises.
- **PC3** (9% of feature variance): USD strength factor. DXY dominates at +0.70. Hershey and VIX positive; oil and gold negative. A strong USD month = strong hazelnut USD price.

In [ ]:
scaler2 = StandardScaler()
X_s2    = scaler2.fit_transform(X_raw)
pca     = PCA(n_components=8)
scores  = pca.fit_transform(X_s2)
loadings = pd.DataFrame(pca.components_.T, index=feat_cols,
                        columns=[f'PC{i+1}' for i in range(8)])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Scree
ax = axes[0]
ve = pca.explained_variance_ratio_ * 100
ax.bar(range(1, len(ve)+1), ve, color='steelblue')
ax.set_xlabel('Component'); ax.set_ylabel('Variance explained (%)')
ax.set_title('Scree plot')

# PC1 loadings
ax = axes[1]
ld1 = loadings['PC1'].sort_values()
ax.barh(ld1.index.str.replace('ret_',''), ld1.values,
        color=['steelblue' if v>0 else 'crimson' for v in ld1])
ax.axvline(0, color='black', lw=0.5)
ax.set_title('PC1 — commodity / risk-on')
ax.set_xlabel('loading')

# PC3 loadings
ax = axes[2]
ld3 = loadings['PC3'].sort_values()
ax.barh(ld3.index.str.replace('ret_',''), ld3.values,
        color=['steelblue' if v>0 else 'crimson' for v in ld3])
ax.axvline(0, color='black', lw=0.5)
ax.set_title('PC3 — USD strength')
ax.set_xlabel('loading')

plt.tight_layout()
plt.show()

## 3 · Which PCs explain hazelnut price? PC1 and PC3.

In [ ]:
results = []
for i in range(6):
    m = _ols(y, np.c_[ones, scores[:,i]], ['int',f'PC{i+1}'], f'PC{i+1}')
    results.append({'PC': f'PC{i+1}', 'var_expl_%': round(ve[i],1),
                    'R2': round(m.rsquared,3), 'adj_R2': round(m.rsquared_adj,3),
                    't': round(m.tvalues[f'PC{i+1}'],2), 'p': round(m.pvalues[f'PC{i+1}'],3)})
rdf = pd.DataFrame(results)
print(rdf.to_string(index=False))

m13 = _ols(y, np.c_[ones, scores[:,0], scores[:,2]], ['int','PC1','PC3'], 'PC1+PC3')
print(f'\nPC1 + PC3 combined:  R2={m13.rsquared:.3f}  adj-R2={m13.rsquared_adj:.3f}  n={m13.n}')
print(pd.DataFrame({'coef':m13.params,'t':m13.tvalues,'p':m13.pvalues}).round(3).to_string())

## 4 · Caveat: PC1+PC3 is period-specific

The full sample is n=207 (2005–2026). The Giresun monthly data only covers 2005–2020 (n=117 overlap). When you refit PC1+PC3 **within that earlier window only**, the result collapses:

| Window | PC1+PC3 R² |
|--------|-----------|
| 2005–2026 full (n=207) | **0.338** |
| 2005–2020 only (n=117) | **0.016** |

The signal is concentrated in 2021–2026 — when the TRY collapsed and Turkish commodity prices became tightly coupled to the global commodity/USD cycle. Before that, hazelnut prices were more locally determined and the global factors didn't apply.

In [ ]:
# Refit on 2005-2020 only to show the period instability
early = base[base['month'] <= '2020-12'].copy()
y_e   = early['ret_vwap_usd'].values
X_e   = StandardScaler().fit_transform(early[feat_cols].values)
pca_e = PCA(n_components=8); sc_e = pca_e.fit_transform(X_e)

m_full  = _ols(y,   np.c_[np.ones(len(y)),   scores[:,0], scores[:,2]],   ['i','PC1','PC3'], 'Full 2005-2026')
m_early = _ols(y_e, np.c_[np.ones(len(y_e)), sc_e[:,0],   sc_e[:,2]],   ['i','PC1','PC3'], 'Early 2005-2020')

print(f'{"Window":<20} {"n":>4} {"R2":>7} {"adj-R2":>8}')
print('-'*44)
for m in [m_full, m_early]:
    print(f'{m.name:<20} {m.n:>4} {m.rsquared:>7.3f} {m.rsquared_adj:>8.3f}')

# Time series showing fitted vs actual across both periods
sub_copy = base.copy()
sub_copy['date']   = pd.to_datetime(sub_copy['month'])
sub_copy['fitted'] = m_full.fitted

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(sub_copy['date'], sub_copy['ret_vwap_usd'], lw=0.8, color='steelblue', alpha=0.7, label='actual')
ax.plot(sub_copy['date'], sub_copy['fitted'],        lw=1.2, color='crimson',                label='PC1+PC3 fitted')
ax.axvline(pd.Timestamp('2021-01-01'), color='gray', ls='--', lw=1, label='2021 (gap start)')
ax.axvline(pd.Timestamp('2023-01-01'), color='gray', ls=':',  lw=1, label='2023 (data resumes)')
ax.axhline(0, color='black', lw=0.4)
ax.set_ylabel('monthly log return (USD/kg)')
ax.set_title('PC1+PC3 fit — most signal comes from 2023–2026 period')
ax.legend(fontsize=8)
import matplotlib.dates as mdates
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

## 5 · Giresun lagged return: small but stable signal

The Giresun exchange (`giresun_spot_prices_monthly.csv`) is a separate, richer dataset with monthly data from 2001–2024. It includes buyer-level breakdown (open market, fisko, TMO) and total volume.

Testing whether **last month's Giresun price or volume** predicts **this month's TOBB all-exchange price**:

| Feature | R² | t | p |
|---------|-----|---|---|
| Lag-1 Giresun USD return | 0.037 | 2.09 | **0.038** |
| Lag-1 Giresun TL return | 0.000 | 0.08 | 0.940 |
| Lag-1 log Giresun volume | 0.006 | 0.80 | 0.427 |
| Lag-1 Giresun volume change | 0.009 | −1.02 | 0.312 |

**Interpretation**: Giresun is the price-discovery benchmark. Its USD price last month has a small predictive edge over the all-exchange aggregate — likely because Giresun leads and Ordu/Düzce follow with a lag. TL price doesn't survive because TRY/USD noise dominates. Volume is not informative.

In [ ]:
# Load Giresun and test lags
g = pd.read_csv('../data/raw/giresun_spot_prices_monthly.csv')
g = g[g['period'] == 'full'].copy()
g['month'] = g['year'].astype(str) + '-' + g['month'].astype(str).str.zfill(2)
g = g.sort_values('month')
g['ret_gir_usd']      = np.log(g['avg_usd_kg_inshell']).diff()
g['log_gir_vol']      = np.log(g['total_qty_kg'])
g['lag1_ret_gir_usd'] = g['ret_gir_usd'].shift(1)
g['lag1_log_gir_vol'] = g['log_gir_vol'].shift(1)
g['lag1_ret_gir_vol'] = g['log_gir_vol'].diff().shift(1)
g = g[['month','lag1_ret_gir_usd','lag1_log_gir_vol','lag1_ret_gir_vol']].dropna()

df = base.merge(g, on='month', how='inner').dropna()
y2   = df['ret_vwap_usd'].values
ones2 = np.ones(len(y2))
print(f'Giresun overlap: n={len(df)}  ({df["month"].iloc[0]} – {df["month"].iloc[-1]})')
print()

rows = []
for col, label in [('lag1_ret_gir_usd','lag Giresun USD ret'),
                   ('lag1_log_gir_vol', 'lag log Giresun vol'),
                   ('lag1_ret_gir_vol', 'lag Giresun vol chg')]:
    m = _ols(y2, np.c_[ones2, df[col].values], ['int',col], col)
    rows.append({'feature': label, 'R2': round(m.rsquared,3),
                 't': round(m.tvalues[col],2), 'p': round(m.pvalues[col],3)})
print(pd.DataFrame(rows).to_string(index=False))

# Scatter: lag Giresun USD vs next month TOBB
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(df['lag1_ret_gir_usd'], y2, s=15, alpha=0.5, color='steelblue')
xs = np.linspace(df['lag1_ret_gir_usd'].min(), df['lag1_ret_gir_usd'].max(), 100)
m_g = _ols(y2, np.c_[ones2, df['lag1_ret_gir_usd'].values], ['int','lag'], 'g')
ax.plot(xs, m_g.params['int'] + m_g.params['lag']*xs, '--', color='crimson', lw=1.2)
ax.set_xlabel('Giresun USD return (month t-1)')
ax.set_ylabel('TOBB all-exchange USD return (month t)')
ax.set_title(f'Lagged Giresun → next month price  (R²={m_g.rsquared:.3f})')
plt.tight_layout()
plt.show()

## Summary

| Model | n | R² | adj-R² | Stable? |
|-------|---|-----|--------|---------|
| Lasso→Ridge (16 features direct) | 207 | ~0 | ~0 | — Lasso zeroes all |
| PC1 only (commodity cycle) | 207 | 0.227 | 0.223 | Partly — driven by post-2021 |
| PC1 + PC3 (commodity + USD) | 207 | 0.338 | 0.332 | Partly — driven by post-2021 |
| PC1 + PC3 (2005–2020 only) | 117 | 0.016 | −0.001 | Not stable pre-2021 |
| Lagged Giresun USD return | 117 | 0.037 | 0.028 | Yes — consistent across periods |

**Why Lasso→Ridge fails**: multicollinearity. All 16 features are correlated with y but also with each other. Lasso can't distinguish signal from redundancy and zeroes everything.

**Why PCA works**: PCA removes multicollinearity by rotating to orthogonal factors. PC1 and PC3 are the two distinct signals in the data.

**The stability caveat**: the PC1+PC3 result is real but concentrated in 2021–2026 — the period of extreme TRY depreciation when Turkish hazelnut prices became tightly coupled to the global commodity/USD cycle. In 2005–2020, global factors explained almost nothing.

**Giresun lead**: small but stable across periods. Giresun sets prices; other exchanges follow.